In [1]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 16 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 

In [13]:
%%writefile cal.l
%{
#include "cal.tab.h"
#include <stdlib.h>
%}

%option noyywrap

%%

[0-9]+(\.[0-9]+)?    {
    yylval = atof(yytext);
    return NUM;
}

[ \t]+               {
    /* ignore spaces */
}

\n                   {
    return '\n';
}

[+\-*/]              {
    return yytext[0];
}

.                    {
    return yytext[0];
}

%%

Overwriting cal.l


In [9]:
%%writefile cal.y
%{
#include <stdio.h>
#include <stdlib.h>

int yylex(void);
int yyerror(const char *s);
%}

%define api.value.type {double}

%token NUM

%left '+' '-'
%left '*' '/'
%right UMINUS

%%

statement:
      E
      {
          printf("Answer: %g\n", $1);
      }
    ;

E:
      E '+' E
      {
          $$ = $1 + $3;
      }
    | E '-' E
      {
          $$ = $1 - $3;
      }
    | E '*' E
      {
          $$ = $1 * $3;
      }
    | E '/' E
      {
          $$ = $1 / $3;
      }
    | NUM
      {
          $$ = $1;
      }
    ;

%%

int main()
{
    printf("Enter the expression:\n");
    yyparse();
    return 0;
}

int yyerror(const char *s)
{
    printf("Syntax Error: %s\n", s);
    return 0;
}

Overwriting cal.y


In [14]:
!rm -f lex.yy.c cal.tab.c cal.tab.h cal

In [15]:
!ls

cal.l  cal.y  sample_data


In [16]:
!bison -d cal.y

In [17]:
!flex cal.l

In [18]:
!gcc lex.yy.c cal.tab.c -o cal -lfl

In [19]:
!echo "2+3" | ./cal

Enter the expression:
Answer: 5
Syntax Error: syntax error
